# 03 — Recursive Prototyping

This notebook walks through the recursive cosine k-means prototyping pipeline (Chapter 5, §5.2).

**Pipeline overview:**
1. **Initial clustering** — FAISS cosine k-means with C=100 clusters on VideoMAE-v2 embeddings (D=1408)
2. **Prototype extraction** — Load centroids, compute RBF distance matrices, assess cluster quality
3. **Manual annotation** — Human expert classifies each prototype (task-definitive / exo-definitive / ambiguous / Noise)
4. **Iterative refinement** — Repeat on residual set (samples far from current prototypes) for P=8 rounds
5. **HAC consolidation** — Hierarchical agglomerative clustering merges validated prototypes (~55 final)
6. **Symbolic representation** — Each temporal segment assigned to its nearest meta-prototype

## 1. Config & Data Loading

Load experiment configuration, video embedding DataFrames, and annotation mappings.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from IPython.display import display
from scipy.spatial.distance import cdist
from sklearn.metrics.pairwise import cosine_similarity

from smartflat.configs.loader import import_config, get_complete_configs
from smartflat.datasets.loader import get_dataset, prepare_training_data
from smartflat.engine.builders import build_model
from smartflat.engine.clustering import predict_with_centroids, main_deployment
from smartflat.features.symbolization.main import (
    build_clusterdf,
    define_symbolization,
    filter_prototypes_per_prevalence,
)
from smartflat.features.symbolization.utils import gaussian_rbf_matrix, get_prototypes
from smartflat.features.symbolization.utils_dataset import get_experiments_dataframe
from smartflat.features.symbolization.co_clustering import clustering_prototypes_space
from smartflat.features.symbolization.inference_reduction_prototypes_distances import (
    main as main_prototypes_reduction,
)
from smartflat.features.symbolization.visualization import (
    plot_cluster_prevalence,
    plot_labels_distributions_subplots,
    plot_centroid_cluster_distances_distributions,
    plot_trajectory,
    plot_ot_graphs,
    visualize_cohort_symbolic_representation,
)
from smartflat.features.symbolization.visualization_prototypes import plot_nearest_centroid_frames
from smartflat.utils.utils_io import get_data_root, fetch_qualification_mapping
from smartflat.utils.utils_visualization import plot_chronogames, plot_gram
from smartflat.utils.utils_dataset import compute_matrix_stats

In [ ]:
# --- Experiment parameters ---
# These define which experiment to inspect. Adjust round_number to explore different iterations.
annotator_id = 'samperochon'
round_number = 8  # Final round (1-8 available)

# Config hierarchy for the recursive loop:
# Round 1: SymbolicSourceFaissCInferenceGoldConfig  (initial FAISS cosine k-means)
# Rounds 2-7: SymbolicSourceInferenceRefinementGoldConfig  (refinement on residuals)
# Round 8 (final): SymbolicSourceInferenceGoldConfig  (inference with definitive prototypes only)

# Configs for inspecting a single round
clustering_config_name = 'ClusteringDeploymentKmeansCombinationCompleteConfig'
symbolization_config_name = 'SymbolicSourceInferenceCompleteGoldConfig'

# Data root
data_root = get_data_root()
print(f"Data root: {data_root}")

In [ ]:
# Load the experiment dataframe for the current round.
# This contains per-participant metadata, embedding labels, distances, and paths.
df = get_experiments_dataframe(
    experiment_config_name=symbolization_config_name,
    annotator_id=annotator_id,
    round_number=round_number,
    return_symbolization=True,
    return_data_only=True,
)
print(f"Loaded {len(df)} recordings")
print(f"Columns: {sorted(df.columns.tolist())[:20]}...")
display(df[['identifier', 'participant_id', 'task_name', 'modality', 'N', 'duration']].head(10))

In [ ]:
# The qualification mapping tracks annotation labels per config/annotator/round.
# Categories: task-definitive, exo-definitive, task-ambiguous, exo-ambiguous, Noise
qualification_mapping = fetch_qualification_mapping(verbose=True)

# Show available annotation rounds
qm_keys = list(qualification_mapping.get(symbolization_config_name, {}).get(annotator_id, {}).keys())
print(f"Available rounds for {annotator_id}: {qm_keys}")

## 2. Initial Clustering (Round 1)

Run cosine k-means with C=100 clusters and d_min=0.3 distance threshold.

See: `smartflat.engine.clustering.main_deployment`, `smartflat.engine.clustering.predict_with_centroids`

In [ ]:
# Round 1 uses FAISS cosine k-means on the full training set
initial_config_name = 'SymbolicSourceFaissCInferenceGoldConfig'
config = import_config(initial_config_name)

print(f"Experiment: {config.experiment_name} / {config.experiment_id}")
print(f"Clustering config: {config.clustering_config_name}")
clustering_config = import_config(config.clustering_config_name)
print(f"Model: {clustering_config.model_name}")
print(f"Model params: {clustering_config.model_params}")

In [ ]:
# Load round-1 experiment dataframe
df_r1 = get_experiments_dataframe(
    experiment_config_name=initial_config_name,
    annotator_id=annotator_id,
    round_number=1,
    return_symbolization=False,
    return_data_only=True,
)

if df_r1 is not None:
    # Build cluster statistics
    clusterdf_r1 = build_clusterdf(
        df_r1,
        embeddings_label_col='raw_embedding_labels',
        segments_labels_col='segments_labels',
        annotator_id=annotator_id,
        round_number=1,
    )
    print(f"Round 1: {len(clusterdf_r1)} clusters, {len(df_r1)} recordings")
    display(clusterdf_r1[['cluster_index', 'cluster_type', 'n_subjects', 'n_appearance',
                           'cluster_dist_mean']].head(10))
else:
    print("Round 1 data not available locally — run on HPC first.")

In [ ]:
if df_r1 is not None:
    # Chronogram: temporal distribution of cluster labels across participants
    plot_chronogames(df_r1, labels_col='raw_embedding_labels', n_subjects=5)
    plt.suptitle('Round 1 — Raw embedding labels (first 5 participants)', y=1.02)
    plt.show()

## 3. Prototype Extraction & Distance Matrices

Load accumulated prototype centroids and compute distance/similarity matrices.

See: `smartflat.features.symbolization.utils.get_prototypes`, `smartflat.features.symbolization.utils.gaussian_rbf_matrix`

In [ ]:
# Load the accumulated prototype centroids (K × D=1408 matrix)
P_all = get_prototypes(
    config_name=symbolization_config_name,
    cluster_types=None,  # All types
    annotator_id=annotator_id,
    round_number=round_number,
    verbose=True,
)
print(f"Prototype matrix shape: {P_all.shape}  (K prototypes × D dimensions)")
compute_matrix_stats(P_all, title='All prototypes')

In [ ]:
# Load prototypes split by annotation category
for cluster_type in ['task-definitive', 'exo-definitive', 'Noise']:
    P_type = get_prototypes(
        config_name=symbolization_config_name,
        cluster_types=[cluster_type],
        annotator_id=annotator_id,
        round_number=round_number,
        verbose=False,
    )
    if P_type is not None and len(P_type) > 0:
        print(f"  {cluster_type}: {P_type.shape[0]} prototypes")

In [ ]:
# Compute Gaussian RBF kernel matrix between prototypes
# sigma = median euclidean distance (~53 for N=168 after Z-norm, cf. constants.MEDIAN_SIGMA_RBF_N168)
from smartflat.constants import MEDIAN_SIGMA_RBF_N168

sigma = MEDIAN_SIGMA_RBF_N168
K_rbf = gaussian_rbf_matrix(P_all, P_all, sigma=sigma, metric='cosine')
print(f"RBF kernel matrix shape: {K_rbf.shape}, range: [{K_rbf.min():.3f}, {K_rbf.max():.3f}]")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cosine_similarity(P_all), cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title(f'Cosine similarity ({P_all.shape[0]}×{P_all.shape[0]})')
axes[1].imshow(K_rbf, cmap='viridis')
axes[1].set_title(f'Gaussian RBF kernel (σ={sigma:.1f})')
plt.tight_layout()
plt.show()

In [ ]:
# Cluster-level distance distributions — shows how tightly samples cluster around prototypes
clusterdf = build_clusterdf(
    df,
    embeddings_label_col='raw_embedding_labels',
    segments_labels_col='segments_labels',
    annotator_id=annotator_id,
    round_number=round_number,
)

# Prevalence filtering: remove clusters with <10 subjects or <10 total occurrences
excluded = filter_prototypes_per_prevalence(clusterdf, min_subjects=10, min_occurences=10)
print(f"Excluded {len(excluded)} low-prevalence clusters")

# Distance distributions for top clusters
plot_centroid_cluster_distances_distributions(
    df,
    labels_col='raw_embedding_labels',
    clustering_config_name=symbolization_config_name,
    title='Distance to nearest prototype per cluster',
)
plt.show()

## 4. Manual Annotation Workflow

Each round's K=100 prototypes are classified by an expert into 5 categories using `pigeon-jupyter`:
**task-definitive**, **exo-definitive**, **task-ambiguous**, **exo-ambiguous**, **Noise**.

See: `smartflat.features.symbolization.main_prototypes_annotation.main`

In [ ]:
# Show the qualification mapping for the current round
qm = qualification_mapping.get(symbolization_config_name, {}).get(annotator_id, {}).get(f'round_{round_number}', {})
for category, indices in qm.items():
    print(f"  {category}: {len(indices)} prototypes — indices {indices[:5]}{'...' if len(indices) > 5 else ''}")

In [ ]:
# Show the nearest video frames to each prototype centroid.
# This is what the annotator sees during the annotation step.
# Note: requires video files on disk. Set show=False to save to disk instead.
plot_nearest_centroid_frames(
    df,
    labels_col='raw_embedding_labels',
    cluster_values=list(range(min(10, P_all.shape[0]))),  # First 10 prototypes
    sampled_index=0,
    show=True,
    n_cols=5,
    title='Nearest frames to prototypes 0-9',
)

### Pigeon annotation (interactive — run on annotation machine only)

The annotation uses `pigeon-jupyter` to classify each prototype. This cell is
**not meant to be run** in this notebook — it documents the interface for reference.

```python
from pigeon import annotate
annotations = annotate(
    image_paths,
    options=['Noise', 'task-definitive', 'task-ambiguous', 'exo-definitive', 'exo-ambiguous'],
    display_fn=lambda path: display(Image(path))
)
```

After annotation, results are saved to:
`{data_root}/dataframes/annotations/pigeon-annotations/{experiment_name}/{experiment_id}/{annotator_id}/round_{N}/`

## 5. Iterative Refinement (P=8 Rounds)

Repeat clustering on the residual set (samples far from current prototypes), annotation, and
prototype accumulation over P=8 rounds.

See: `smartflat.configs.loader.get_complete_configs`, `smartflat.features.symbolization.main.define_symbolization`

In [ ]:
# The recursive loop builds a chain of configs, each referencing the previous round's outputs.
# get_complete_configs() reconstructs this chain for a given target round.
input_configs, round_numbers_list, qual_types = get_complete_configs(round_number=round_number)

print("Recursive config chain:")
print(f"{'Round':<8} {'Config':<55} {'Qualified types'}")
print("-" * 110)
for r, cfg, qt in zip(round_numbers_list, input_configs, qual_types):
    print(f"{r:<8} {cfg:<55} {qt}")

In [ ]:
# Load cluster statistics for each available round and track how prototypes evolve
round_stats = []
for r in range(1, round_number + 1):
    try:
        df_r = get_experiments_dataframe(
            experiment_config_name='SymbolicSourceInferenceCompleteGoldConfig',
            annotator_id=annotator_id,
            round_number=r,
            return_symbolization=False,
            return_data_only=True,
        )
        if df_r is not None:
            n_labels = len(np.unique(np.hstack(df_r['raw_embedding_labels'])))
            mean_dist = np.mean(np.hstack(df_r['cluster_dist']))
            round_stats.append({'round': r, 'n_prototypes': n_labels, 'mean_distance': mean_dist, 'n_recordings': len(df_r)})
    except Exception as e:
        print(f"  Round {r}: not available ({e})")

if round_stats:
    stats_df = pd.DataFrame(round_stats)
    display(stats_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(stats_df['round'], stats_df['n_prototypes'], 'o-')
    axes[0].set(xlabel='Round', ylabel='# Prototypes', title='Prototype accumulation across rounds')
    axes[1].plot(stats_df['round'], stats_df['mean_distance'], 'o-', color='tab:orange')
    axes[1].set(xlabel='Round', ylabel='Mean distance to centroid', title='Residual distance reduction')
    plt.tight_layout()
    plt.show()

### The recursive cycle (per round)

Each round follows 4 steps:

1. **Clustering** (`main_deployment`): Run cosine k-means (C=100) on the current residual set
2. **Symbolization** (`define_symbolization`): Assign all samples to nearest prototype, compute distances
3. **Visualization** (`plot_nearest_centroid_frames`): Generate prototype images for annotation
4. **Annotation** (pigeon-jupyter): Human expert classifies prototypes → feeds into next round

The residual set for round *r+1* consists of samples whose distance to the nearest
round-*r* prototype exceeds a threshold (d > mean + n_sigma × std).

In [ ]:
# Demonstrate running one iteration of the recursive loop.
# Set overwrite=False to load pre-computed results. Set to True to recompute (slow, needs HPC).
OVERWRITE = False
DEMO_ROUND = round_number  # Which round to inspect

# Step 1: Clustering deployment (uncomment to recompute)
# outputs = main_deployment(clustering_config_name, annotator_id, DEMO_ROUND, overwrite=OVERWRITE)

# Step 2: Symbolization
df_demo, clusterdf_demo, prototypes_demo, qm_demo, results_demo, best_df_demo, \
    mapping_K, mapping_G, excluded_demo, info_demo = define_symbolization(
    symbolization_config_name=symbolization_config_name,
    annotator_id=annotator_id,
    round_number=DEMO_ROUND,
    overwrite_inference=False,
    do_save=False,
    reset_init_build=True,
    do_compute_clustering=False,
    clustering_input_spaces=['K_space', 'G_space'],
    overwrite_distance_thresholds=False,
    overwrite=False,
    verbose=True,
)

In [ ]:
if df_demo is not None:
    print(f"Recordings: {len(df_demo)}")
    print(f"Prototype categories: { {k: len(v) if isinstance(v, np.ndarray) else 'N/A' for k, v in prototypes_demo.items()} }")
    print(f"K-space mapping: {len(mapping_K)} source → meta prototypes")
    print(f"G-space mapping: {len(mapping_G)} meta → G prototypes" if mapping_G else "G-space: identity")
    print(f"Excluded clusters: {excluded_demo}")
    display(clusterdf_demo[['cluster_index', 'cluster_type', 'n_subjects', 'n_appearance']].head(10))

## 6. HAC Consolidation

Hierarchical agglomerative clustering (Ward linkage) merges K source prototypes into ~G
meta-prototypes, optimizing silhouette score and Davies-Bouldin index.

See: `smartflat.features.symbolization.co_clustering.clustering_prototypes_space`

In [ ]:
# HAC consolidation reduces K source prototypes into G meta-prototypes.
# Uses hierarchical Ward linkage on the prototype embedding space,
# optimizing silhouette score and Davies-Bouldin index.

results, best_df, mapping_prototypes_reduction, mapping_index_cluster_type = clustering_prototypes_space(
    symbolization_config_name=symbolization_config_name,
    model_names=['hierarchical-ward'],
    annotator_id=annotator_id,
    input_space='K_space',
    round_number=round_number,
    verbose=True,
    do_save=False,
    overwrite=False,
)

print(f"\nBest HAC solution:")
display(best_df)
print(f"\nMapping K→G: {len(mapping_prototypes_reduction)} entries")
print(f"Unique G clusters: {len(set(mapping_prototypes_reduction.values()))}")

In [ ]:
# Show how source prototypes map to meta-prototypes
mapping_df = pd.DataFrame([
    {'source_K': k, 'meta_G': v, 'cluster_type': mapping_index_cluster_type.get(v, 'Unknown')}
    for k, v in mapping_prototypes_reduction.items()
])
print(f"Source prototypes: {len(mapping_df)}")
print(f"Meta-prototypes: {mapping_df['meta_G'].nunique()}")
print(f"\nCluster type distribution:")
display(mapping_df.groupby('cluster_type')['meta_G'].nunique().to_frame('n_meta_prototypes'))

# Histogram of source-to-meta grouping sizes
fig, ax = plt.subplots(figsize=(10, 4))
mapping_df.groupby('meta_G').size().hist(bins=20, ax=ax)
ax.set(xlabel='# source prototypes per meta-prototype', ylabel='Count',
       title='HAC consolidation: source → meta prototype grouping')
plt.tight_layout()
plt.show()

In [ ]:
# Reduce per-sample distance matrices from K-space to G-space
# using the HAC mapping (aggregation by min distance within each meta-prototype group)
if df_demo is not None:
    df_reduced = main_prototypes_reduction(
        config_name=symbolization_config_name,
        df=df_demo,
        annotator_id=annotator_id,
        round_number=round_number,
        input_space='K_space',
        mapping_prototypes_reduction=mapping_prototypes_reduction,
        overwrite=False,
        verbose=True,
    )
    print(f"Distance reduction complete — {len(df_reduced)} recordings processed")

## 7. Results Visualization

Visualize the final symbolic representation: prototype trajectories, label distributions,
cohort-level patterns, and per-participant similarity matrices.

See: `smartflat.features.symbolization.visualization`, `smartflat.utils.utils_visualization`

In [ ]:
# Final symbolic representation: per-participant temporal label sequences
if df_demo is not None:
    plot_chronogames(df_demo, labels_col='opt_embedding_labels', n_subjects=8)
    plt.suptitle(f'Final symbolic labels (round {round_number}, K-space)', y=1.02)
    plt.show()

In [ ]:
if df_demo is not None:
    plot_labels_distributions_subplots(
        df_demo,
        clustering_config_names=[symbolization_config_name],
        labels_col='opt_embedding_labels',
        segments_labels_col='opt_segments_labels',
        clusterdf=clusterdf_demo,
    )
    plt.show()

In [ ]:
if df_demo is not None:
    plot_cluster_prevalence(
        df_demo,
        segments_labels_col='opt_segments_labels',
        title=f'Prototype prevalence (round {round_number})',
        verbose=True,
    )
    plt.show()

In [ ]:
if df_demo is not None:
    visualize_cohort_symbolic_representation(
        df_demo,
        covariate='n_cluster',
        labels_col='opt_embedding_labels',
    )
    plt.suptitle(f'Cohort symbolic representation (round {round_number})', y=1.02)
    plt.show()

In [ ]:
if df_demo is not None:
    row = df_demo.iloc[0]
    plot_gram(
        row,
        temporal_segmentation_col='opt_cpts',
        segments_labels_col='opt_segments_labels',
        title=f'{row.identifier} — Gram matrix with symbolic segments',
    )
    plt.show()